## Goal: Prepare our feature columns for the model
* handle missing values
* convert categorical columns to numerical values
* remove any extraneous columns througout

In [ ]:
import pandas as pd
loans = pd.read_csv("filtered_loans_2007.csv")

print(loans.head(1))
print(loans.info())

   Unnamed: 0  loan_amnt        term int_rate  installment emp_length  \
0           0     5000.0   36 months   10.65%       162.87  10+ years   

  home_ownership  annual_inc verification_status  loan_status  ...  \
0           RENT     24000.0            Verified            1  ...   

  delinq_2yrs earliest_cr_line inq_last_6mths  open_acc  pub_rec revol_bal  \
0         0.0         Jan-1985            1.0       3.0      0.0   13648.0   

   revol_util  total_acc  last_credit_pull_d  pub_rec_bankruptcies  
0       83.7%        9.0            Jun-2016                   0.0  

[1 rows x 24 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24909 entries, 0 to 24908
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            24909 non-null  int64  
 1   loan_amnt             24909 non-null  float64
 2   term                  24909 non-null  object 
 3   int_rate              249

In [ ]:
null_counts = loans.isnull().sum()
print(null_counts[null_counts > 0])

emp_length              857
title                     1
revol_util               14
total_acc                 1
last_credit_pull_d        1
pub_rec_bankruptcies      1
dtype: int64




* **emp_length**          | object  | Employment length in years. Possible values are between 0 and 10 where 0 means less than one year and 10 means ten or more years.
* **pub_rec_bankruptcies**       | float64 | Number of public record bankruptcies


In [ ]:
print(loans.pub_rec_bankruptcies.value_counts(normalize=True, dropna=False))

0.0    0.955358
1.0    0.044442
2.0    0.000161
NaN    0.000040
Name: pub_rec_bankruptcies, dtype: float64


## Dropping 'pub_rec_bankruptcies'
Very little variablility in 'pub_rec_bankruptcies' means we can safely drop the column.

We will drop the rows in the other columns with NaN values,

In [ ]:
loans = loans.drop(columns="pub_rec_bankruptcies")

In [ ]:
# Drop all columns with ANY missing values
# drop_na_cols = ['emp_length', 'title', 'revol_util', 'last_credit_pull_d']

loans = loans.dropna()  # subset=drop_na_cols)
# this is the default nature of .dropna() anyway, so a subset isn't necessary

loans.dtypes.value_counts()

object     11
float64    10
int64       2
dtype: int64

In [ ]:
# what other feature columns need cleaning?
object_columns_df = loans.select_dtypes(include=['object'])
print(object_columns_df.head(1))

         term int_rate emp_length home_ownership verification_status  \
0   36 months   10.65%  10+ years           RENT            Verified   

       purpose     title addr_state earliest_cr_line revol_util  \
0  credit_card  Computer         AZ         Jan-1985      83.7%   

  last_credit_pull_d  
0           Jun-2016  


## Text Columns that require conversion
### Categorical Columns
* **home_ownership**: home ownership status, can only be 1 of 4 categorical values according to the data dictionary
* **verification_status**: indicates if income was verified by Lending Club
* **emp_length**: number of years the borrower was employed upon time of application
* **term**: number of payments on the loan, either 36 or 60
* **addr_state**: borrower's state of residence
* **purpose**: a category provided by the borrower for the loan request
* **title**: loan title provided by the borrower
**NOTE:** purpose and title columns could reflect the same information

### Numerical columns
* **int_rate**: interest rate of the loan in %
* **revol_util: revolving line utilization rate or the amount of credit the borrower is using relative to all available credit, read more [here](http**://blog.credit.com/2013/04/what-is-revolving-utilization-65530/)

### Date columns
Requires significant feature engineering to be useful
* **earliest_cr_line**: The month the borrower's earliest reported credit line was opened
* **last_credit_pull_d**: The most recent month Lending Club pulled credit for this loan

In [ ]:
# First 5 Categorical Columns
cols = ['home_ownership', 'verification_status', 'emp_length', 'term', 'addr_state']

for c in cols:
    print(loans[c].value_counts())

RENT        11191
MORTGAGE    11140
OWN          1707
OTHER           1
Name: home_ownership, dtype: int64
Verified           8400
Not Verified       7878
Source Verified    7761
Name: verification_status, dtype: int64
10+ years    5937
< 1 year     2552
3 years      2523
2 years      2479
4 years      2174
5 years      2170
1 year       1788
6 years      1561
7 years      1160
8 years       925
9 years       770
Name: emp_length, dtype: int64
 36 months    16106
 60 months     7933
Name: term, dtype: int64
CA    4321
NY    2266
FL    1724
TX    1665
NJ    1108
IL     934
PA     885
VA     808
GA     807
OH     742
MA     735
NC     686
MD     618
AZ     520
WA     511
CT     451
CO     439
MO     437
MI     413
MN     369
NV     333
SC     292
LA     284
WI     268
AL     266
OR     262
KY     218
KS     197
OK     196
AR     161
UT     155
DC     128
RI     123
HI     117
NM     107
WV     103
NH      98
DE      67
AK      53
MT      49
SD      43
WY      40
VT      38
MS       1
TN 

In [ ]:
print(loans['title'].value_counts())
print(loans['purpose'].value_counts())

Debt Consolidation                          1570
Debt Consolidation Loan                     1479
Personal Loan                                392
debt consolidation                           372
Consolidation                                351
                                            ... 
Debt And Credit Card Reduction                 1
Home Loan-Credit Card Consolidation            1
Swap this loan for another loan same amt       1
PAYOFDEBT                                      1
Sadie                                          1
Name: title, Length: 10556, dtype: int64
debt_consolidation    11823
credit_card            3153
other                  1948
home_improvement       1854
major_purchase         1264
car                    1045
small_business         1043
wedding                 578
medical                 440
moving                  343
vacation                251
house                   220
renewable_energy         71
educational               6
Name: purpose, dtype: int64


## Cleaning 'purpose' & 'title'
It seems 'purpose' and 'title' columns do contain overlapping information.

'purpose' column contains a few discrete values

'title' column has data quality issues since many of the values are repeated with slight modifications (e.g. Debt Consolidation and Debt Consolidation Loan and debt consolidation).

In [ ]:
# 'addr_state' contains too many discrete values & would bloat the dataset.
# removing 'addr_state'
# dropping other columns
drop_cols = ['last_credit_pull_d', 'addr_state', 'title', 'earliest_cr_line']
loans = loans.drop(columns=drop_cols)

In [ ]:
# cleaning 'revol_util' & 'int_rate'
loans['revol_util'] = loans['revol_util'].str.rstrip(to_strip='%').astype('float')
loans['int_rate'] = loans['int_rate'].str.rstrip(to_strip='%').astype('float')

In [ ]:
# cleaning 'emp_length'
# simply mapped the text to the year count numeric
# rough approximations: 10+ maps to 10,   < 1 years maps to 0,   "n/a" maps to  0
mapping_dict = {
    "emp_length": {
        "10+ years": 10,
        "9 years": 9,
        "8 years": 8,
        "7 years": 7,
        "6 years": 6,
        "5 years": 5,
        "4 years": 4,
        "3 years": 3,
        "2 years": 2,
        "1 year": 1,
        "< 1 year": 0,
        "n/a": 0
    }
}

loans = loans.replace(mapping_dict)

In [ ]:
# convert categorical columns into numeric values (for model)
# Returns a new Dataframe containing 1 column for each dummy variable.
cols_to_category = ["home_ownership", "verification_status", "purpose", "term"]
dummy_df = pd.get_dummies(loans[cols_to_category])

loans = pd.concat([loans, dummy_df], axis='columns')

loans = loans.drop(columns=cols_to_category)

In [ ]:
loans.to_csv("clean_loans_2007.csv")